# Image Alignment and Stacking

The telescope does not track perfectly. Over a set of exposures your target
drifts across the detector, so the frames cannot simply be added together --
first they have to be *aligned* to a common reference.

This notebook covers that last step of data reduction:

1. Measure how far the target has drifted between frames.
2. Shift each frame back onto a reference frame.
3. Sum the aligned frames into a single master frame with higher
   signal-to-noise than any individual exposure.
4. Save that master frame, with its header, to a new FITS file.

The worked example uses observations of the binary star Albireo in the V
filter; the exercise at the end asks you to repeat it for B, R, and I.

## Before you start: reduced frames

Alignment happens *after* dark subtraction and flat fielding. This notebook
assumes you already have reduced frames, which is what
[the data reduction exercise](DataReduction_SingleFrame_exercise.ipynb)
walks you through.

The cell below is a condensed recap of that reduction so this notebook runs on
its own -- it is not the lesson here. Point `data_dir` at your own data and
adjust the exposure times to match what you actually observed with.

In [ ]:
import glob

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval, ImageNormalize

# Where your data lives. Adjust these to match your own directory layout.
data_dir = './Data/'
target = 'Albireo'
filter_name = 'V'

science_files = sorted(glob.glob(f'{data_dir}{target}/{filter_name}/*.fit'))
dark_files = sorted(glob.glob(f'{data_dir}Darks/*.fit'))
flat_files = sorted(glob.glob(f'{data_dir}Flats/{filter_name}/*.fit'))

print(f"{len(science_files)} science, {len(dark_files)} dark, {len(flat_files)} flat frames")

In [ ]:
# --- condensed recap of the reduction; see the reduction notebook for the why ---

def master_dark_for(exptime):
    """Median-combine the dark frames matching a given exposure time."""
    frames = [fits.getdata(f).astype(float) for f in dark_files
              if fits.getheader(f)['EXPTIME'] == exptime]
    if not frames:
        raise ValueError(f"no dark frames at EXPTIME={exptime}")
    return np.median(np.array(frames), axis=0)

science_exptime = fits.getheader(science_files[0])['EXPTIME']
flat_exptime = fits.getheader(flat_files[0])['EXPTIME']

# Master flat: dark-subtract each flat, median-combine, then normalize by the mean
flats = [fits.getdata(f).astype(float) - master_dark_for(flat_exptime)
         for f in flat_files]
master_flat = np.median(np.array(flats), axis=0)
master_flat_normalized = master_flat / np.mean(master_flat)

# Reduced science frames: dark-subtract, then divide by the normalized flat
science_dark = master_dark_for(science_exptime)
albireo_V_reduced = [(fits.getdata(f).astype(float) - science_dark) / master_flat_normalized
                     for f in science_files]

# One shared scaling, so every image below is displayed on the same stretch
reduced_image_norm = ImageNormalize(albireo_V_reduced[0], interval=ZScaleInterval())

print(f"{len(albireo_V_reduced)} reduced frames, "
      f"{science_exptime} s each, flats at {flat_exptime} s")

## Measuring the drift

First, let's see how much Albireo drifts in the V filter. We find the (X, Y)
coordinate of the brightest pixel in the first image and use that as our
reference point.

Using the brightest pixel works here because Albireo is bright and isolated.
It is a crude method -- for a crowded field, or a target that is not the
brightest thing in the frame, you would want to centroid on a chosen star
instead.

In [ ]:
# Define a reference frame from the reduced Albireo V images
reference_frame = albireo_V_reduced[0]

# Calculate the x,y pixel position of the brightest point in the reference image
reference_ypix, reference_xpix = np.unravel_index(reference_frame.argmax(), reference_frame.shape)

# Plot the reference image and mark the position of the brightest pixel
plt.imshow(reference_frame, origin='lower', cmap='gray', norm=reduced_image_norm)
plt.scatter(reference_xpix, reference_ypix, s=100, color='red', marker='x')
plt.title('Reference Frame with Brightest Pixel Marked')

plt.ylim(reference_ypix - 150, reference_ypix + 150)
plt.xlim(reference_xpix - 150, reference_xpix + 150)

plt.show()

As expected, the brightest pixel is around the center of the brightest star in Albireo. Now, let's see how much the stars drift relative to this reference point.

In [ ]:
# Define the target image list
target_images = albireo_V_reduced[1:]  

# Plot all the target images with the reference point marked using a 3x3 grid of subplots
fig, axes = plt.subplots(3, 3, figsize=(15, 15), tight_layout=True)
axes = axes.ravel()  # Flatten the 2D array of axes to 1

for i, ax in enumerate(axes):
    ax.imshow(target_images[i], origin='lower', cmap='gray', norm=reduced_image_norm)
    ax.scatter(reference_xpix, reference_ypix, s=100, color='red', marker='x')
    ax.set_title(f'Target Image {i+1}')
    ax.set_ylim(reference_ypix - 150, reference_ypix + 150)
    ax.set_xlim(reference_xpix - 150, reference_xpix + 150)

plt.show()


From the plots above, you can see that the stars seem to be shifting up and to the left slightly. Since the exposure time used to collect these images was very low, you shouldn't expect that much drifting, but for targets that require a longer exposure time, you can expect larger drifts, as well as having sources that do not look like *point sources*.

Let's align each individual image now and re-plot the images relative to the brightest pixel of the reference image

## Aligning the frames

`np.roll` shifts an array by a whole number of pixels, wrapping around at the
edges. That wrap-around is why a few rows and columns at the edge of an aligned
frame contain data from the opposite edge -- harmless as long as your target is
not near the border.

In [ ]:
# Define a list to add the aligned images to
aligned_images = []

# Plot all the target images with the reference point marked using a 3x3 grid of subplots
fig, axes = plt.subplots(3, 3, figsize=(15, 15), tight_layout=True)
axes = axes.ravel()  # Flatten the 2D array of axes to 1

# Loop over each target image to align it to the reference frame
for i, target_image in enumerate(target_images):
    # Calculate the (y, x) position of the brightest pixel in the target image
    target_ypix, target_xpix = np.unravel_index(target_image.argmax(), target_image.shape)
    
    # Calculate the shift needed to align the target image to the reference frame
    y_shift = reference_ypix - target_ypix
    x_shift = reference_xpix - target_xpix
    
    # Apply the shift to the target image using np.roll
    aligned_image = np.roll(target_image, shift=(y_shift, x_shift), axis=(0, 1))
    
    # Append the aligned image to the list
    aligned_images.append(aligned_image)

    # Plot the aligned image with the reference point marked
    ax = axes[i]
    ax.imshow(aligned_image, origin='lower', cmap='gray', norm=reduced_image_norm)
    ax.scatter(reference_xpix, reference_ypix, s=100, color='red', marker='x')
    ax.set_title(f'Aligned Image {i+1}')
    ax.set_ylim(reference_ypix - 150, reference_ypix + 150)
    ax.set_xlim(reference_xpix - 150, reference_xpix + 150)
plt.show()


Now, you can see that the reference point is roughly about in the same (X,Y) pixel position for all the stars. 

Once all the images are aligned for the V filter, we can create our master frame called Albireo V Master Frame. This is the final frame and the last step for reducing the frames for the specific filter.

In [ ]:
# Add the reference frame to the list of aligned images and median combine them into the master frame
aligned_images.append(reference_frame)

albireo_V_master_frame = np.sum(np.array(aligned_images), axis=0)

# Normalize the master frame to ZScale for better visualization
albireo_V_master_frame_norm = ImageNormalize(albireo_V_master_frame, interval=ZScaleInterval())

# Display the Albireo V Master Frame
plt.title('Albireo V Master Frame') 
plt.imshow(albireo_V_master_frame, origin='lower', cmap='gray', norm=albireo_V_master_frame_norm)
plt.colorbar(label='Counts [ADU/pixel]')
plt.show()

From this master frame, the total counts increased relative to a single
reduced frame. That is how you achieve better signal-to-noise on your image,
and therefore more precise photometry.

## Saving the master frame to a `.fits` file

Finally, save the master frame to your directory. Include the header
information when you save. Since the master frame is a combination of multiple
images, you can use the header from the reference image -- you need to extract
it from the raw data.

In [ ]:
# Get the header information from the reference image
header_info = fits.getheader(science_files[0])

# Where to save it, and under what name
save_dir = './'  # change this to your desired directory
output_file = f'{target}_{filter_name}_Master_Frame.fits'

# Build the file with the master frame data and the header, and record the filter
hdu = fits.PrimaryHDU(data=albireo_V_master_frame, header=header_info)
hdu.header['FILTER'] = (filter_name, 'Filter used during observation')

# Write the master frame to a new .fits file
hdu.writeto(save_dir + output_file, overwrite=True)

And that's it -- those are the basic steps to align and stack a set of frames.

**Note:** when you use your reduced data to perform photometry, or to create a
colour image, make sure the master images are also aligned *to one another*
across filters (Master B, V, R, and I for Albireo all need to be aligned with
each other). Do this by repeating the alignment process above, picking one
master frame as the reference and treating the others as target frames.

# Student Exercise

You were given observations of Albireo in the BVRI filters. The V filter is
worked through above. Your job is to align and stack the observations for the
remaining filters (B, R, and I).

For each filter you should:

1. Load your reduced frames for that filter. (If you have not reduced them
   yet, do that first -- see the reduction notebook.)
2. Show the drift before alignment: mark the reference frame's brightest pixel
   on each individual frame, so the shift is visible.
3. Align each frame to the reference and re-plot, showing that the drift is
   gone.
4. Sum the aligned frames into a master frame and display it.
5. Save each master frame as a `.fits` file with the header from its reference
   image. Show the code you used; you do **not** need to submit the `.fits`
   files themselves.

Complete these steps in the cells below. To add more cells, press `ESC + A` or
`ESC + B` for a new cell above or below the current one, and `ESC + X` to
delete one.

## Align Albireo Images: B Filter

## Align Albireo Images: R Filter

## Align Albireo Images: I Filter

# Extra Credit Opportunity (5%)

For extra credit you can take your master science frames in the B, V, and R filters and combine them to create a composite color image of this binary star system. Remember that the Master Frames from each filter MUST first be aligned with each other. Then you can then use the following code to create a color image:

```Python
def image_scaling(image, brightness=0.0, contrast=1.0, stretch=1.0):
    
    """
    Scale and adjust the brightness, contrast, and stretch of a 2D image array.

    Parameters:
    - image (ndarray): 2D array representing the image.
    - brightness (float): Brightness adjustment factor. Positive values increase brightness, negative values decrease brightness.
    - contrast (float): Contrast adjustment factor. Multiplies the normalized image data to increase or decrease contrast.
    - stretch (float): Stretch adjustment factor. Exponent applied to the normalized and adjusted image data to further enhance or compress contrast.

    Returns:
    - ndarray: Final image array after applying brightness, contrast, and stretch adjustments.

    Notes:
    - This function first normalizes the input image to the vmax level based on the ZScale scale.
    - Then, it applies the provided brightness, contrast, and stretch adjustments to the normalized image.
    - The formula used for adjustment is ((contrast * normalized_image) + brightness)**stretch.
    - Adjustments are applied linearly for brightness and contrast, and non-linearly for stretch.

    Example:
    >>> final_image = image_scaling(image_data, brightness=0.1, contrast=1.2, stretch=0.8)
    """
    
    # Find the vmin and vmax values of the image based on the XScale scale
    zscale_norm = ImageNormalize(image, interval=ZScaleInterval())
    vmin, vmax = zscale_norm.vmin, zscale_norm.vmax
    
    # Normalize the image to the vmax level
    image_norm = image / vmax
    
    # Add brightness, contrast, and stretch to the normalized image and create the final image
    final_image = ((contrast * image_norm) + brightness)**stretch
    
    return final_image

# Scale the master images
scaled_albireo_master_R = image_scaling(albireo_master_R, -1.2, 1.7, 1.0) # Change these values accordingly until you get an image that yor are happy with
scaled_albireo_master_V = image_scaling(albireo_master_V, -1.2, 1.7, 1.0)
scaled_albireo_master_B = image_scaling(albireo_master_B, -1.2, 1.7, 1.0)

albireo_rgb = np.dstack((scaled_albireo_master_R, scaled_albireo_master_V, scaled_albireo_master_B)) 

# Display the final re-scaled Albireo RGB image
plt.imshow(albireo_rgb)

plt.title('Albireo')
plt.axis('off')

plt.show()
```

If you cannot get this to work, don't worry. We will learn more about RGB images later